In [1]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Enable autoreload for development (automatically reloads modules when they change)
%load_ext autoreload
%autoreload 2

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

# Use CuPy backend for GPU acceleration (falls back to NumPy if not available)
from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n import BeliefMDP_n
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Verification tests for F function (filter update) in BeliefMDP_n.

Tests:
1. F, F_batch, and F_batch_log comparison
2. Numerical agreement validation
3. Performance comparison

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


'\nVerification tests for F function (filter update) in BeliefMDP_n.\n\nTests:\n1. F, F_batch, and F_batch_log comparison\n2. Numerical agreement validation\n3. Performance comparison\n\nUses the same model as T_mat_visuals.ipynb:\n- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0\n- LIDAR(fov=360, r_max=10.0, B=8)\n'

In [2]:
def test_F_batch_comparison(quantization_level=2, n_observations=1000, batch_sizes=[1, 10, 50, 100, 200, 500]):
    """
    Compare F, F_batch, and F_batch_log implementations.
    
    Tests:
    1. Numerical agreement between all three methods
    2. Performance comparison
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        n_observations: Number of observations to test
        batch_sizes: List of batch sizes to test for batched methods
    """
    
    obstacles, area = load_obstacles_config(environment='toy2')
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,
    )
    bmdp = BeliefMDP_n(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)
    
    print(f"\n{'='*70}")
    print(f"=== F, F_batch, and F_batch_log Comparison Test ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"Number of observations: {n_observations}")
    print(f"Batch sizes to test: {batch_sizes}")
    print(f"{'='*70}\n")
    
    # Create test belief
    m_n = bmdp.SQ.m_n
    M_size = bmdp.len_M
    
    # Uniform prior
    π_0 = np.ones((m_n, M_size), dtype=np.float64) / (m_n * M_size)
    
    u = np.array([0.0, 0.0])
    
    # Generate test observations
    print("Generating test observations...")
    Y_samples = []
    for _ in range(n_observations):
        y_sample = random.uniform(0.1, sensor.r_max, sensor.B)
        Y_samples.append(y_sample)
    Y_batch_all = np.array(Y_samples)  # (n_observations, B) - keep on GPU
    
    results = {}
    
    # Test 1: Sequential F (baseline)
    print(f"\n--- Testing Sequential F (one observation at a time) ---")
    π_F_results = []
    start_time = time.time()
    for y_sample in tqdm(Y_samples, desc="Sequential F", leave=False):
        π_new = bmdp.F(π_0, u, y_sample)
        π_F_results.append(π_new)
    time_F = time.time() - start_time
    π_F_results = np.array(π_F_results)  # (n_observations, m_n, len_M)
    
    results['F'] = {
        'time': time_F,
        'time_per_call': time_F / n_observations * 1000,
        'beliefs': π_F_results
    }
    print(f"  Time: {time_F:.4f}s ({results['F']['time_per_call']:.4f} ms/call)")
    
    # Test 2: F_batch with different batch sizes
    for batch_size in batch_sizes:
        if batch_size > n_observations:
            continue
            
        print(f"\n--- Testing F_batch (batch_size={batch_size}) ---")
        π_F_batch_results = []
        n_batches = (n_observations + batch_size - 1) // batch_size
        
        start_time = time.time()
        for i in tqdm(range(n_batches), desc=f"F_batch (size={batch_size})", leave=False):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, n_observations)
            Y_batch = Y_batch_all[start_idx:end_idx]  # (current_batch_size, B)
            
            π_new_batch = bmdp.F_batch(π_0, u, Y_batch)  # (current_batch_size, m_n, len_M)
            
            # Convert to CPU only once per batch
            π_new_batch_cpu = π_new_batch.get() if is_cupy else π_new_batch
            π_F_batch_results.extend(π_new_batch_cpu)
        
        time_F_batch = time.time() - start_time
        π_F_batch_results = np.array(π_F_batch_results)  # (n_observations, m_n, len_M)
        
        results[f'F_batch_{batch_size}'] = {
            'time': time_F_batch,
            'time_per_call': time_F_batch / n_observations * 1000,
            'time_per_batch': time_F_batch / n_batches * 1000,
            'beliefs': π_F_batch_results
        }
        speedup = time_F / time_F_batch
        print(f"  Time: {time_F_batch:.4f}s ({results[f'F_batch_{batch_size}']['time_per_call']:.4f} ms/call)")
        print(f"  Speedup: {speedup:.2f}x vs sequential")
        print(f"  Time per batch: {results[f'F_batch_{batch_size}']['time_per_batch']:.4f} ms")
    
    # Test 3: F_batch_log with different batch sizes
    for batch_size in batch_sizes:
        if batch_size > n_observations:
            continue
            
        print(f"\n--- Testing F_batch_log (batch_size={batch_size}) ---")
        π_F_batch_log_results = []
        n_batches = (n_observations + batch_size - 1) // batch_size
        
        start_time = time.time()
        for i in tqdm(range(n_batches), desc=f"F_batch_log (size={batch_size})", leave=False):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, n_observations)
            Y_batch = Y_batch_all[start_idx:end_idx]  # (current_batch_size, B)
            
            π_new_batch = bmdp.F_batch_log(π_0, u, Y_batch)  # (current_batch_size, m_n, len_M)
            
            # Convert to CPU only once per batch
            π_new_batch_cpu = π_new_batch.get() if is_cupy else π_new_batch
            π_F_batch_log_results.extend(π_new_batch_cpu)
        
        time_F_batch_log = time.time() - start_time
        π_F_batch_log_results = np.array(π_F_batch_log_results)  # (n_observations, m_n, len_M)
        
        results[f'F_batch_log_{batch_size}'] = {
            'time': time_F_batch_log,
            'time_per_call': time_F_batch_log / n_observations * 1000,
            'time_per_batch': time_F_batch_log / n_batches * 1000,
            'beliefs': π_F_batch_log_results
        }
        speedup = time_F / time_F_batch_log
        print(f"  Time: {time_F_batch_log:.4f}s ({results[f'F_batch_log_{batch_size}']['time_per_call']:.4f} ms/call)")
        print(f"  Speedup: {speedup:.2f}x vs sequential")
        print(f"  Time per batch: {results[f'F_batch_log_{batch_size}']['time_per_batch']:.4f} ms")
    
    # Numerical agreement check
    print(f"\n{'='*70}")
    print("Numerical Agreement:")
    π_F_ref = results['F']['beliefs']
    
    # Check F_batch vs F
    for batch_size in batch_sizes:
        if batch_size > n_observations:
            continue
        key = f'F_batch_{batch_size}'
        π_F_batch = results[key]['beliefs']
        
        # Compute relative errors for each belief
        max_rel_error = 0.0
        for i in range(n_observations):
            π_ref = π_F_ref[i]
            π_test = π_F_batch[i]
            
            # Avoid division by zero
            threshold = 1e-100
            non_zero_mask = (np.abs(π_ref) > threshold) | (np.abs(π_test) > threshold)
            if np.any(non_zero_mask):
                denom = np.maximum(np.abs(π_ref[non_zero_mask]), np.abs(π_test[non_zero_mask]))
                denom = np.maximum(denom, threshold)
                rel_errors = np.abs(π_ref[non_zero_mask] - π_test[non_zero_mask]) / denom
                max_rel_error = max(max_rel_error, float(np.max(rel_errors)))
        
        print(f"  F_batch (size={batch_size}) vs F: max rel error = {max_rel_error:.6e}")
        if max_rel_error < 1e-10:
            print(f"    ✓ Excellent agreement")
        elif max_rel_error < 1e-6:
            print(f"    ✓ Good agreement")
        elif max_rel_error < 1e-3:
            print(f"    ⚠ Acceptable agreement")
        else:
            print(f"    ✗ Significant disagreement")
    
    # Check F_batch_log vs F
    for batch_size in batch_sizes:
        if batch_size > n_observations:
            continue
        key = f'F_batch_log_{batch_size}'
        π_F_batch_log = results[key]['beliefs']
        
        # Compute relative errors for each belief
        max_rel_error = 0.0
        for i in range(n_observations):
            π_ref = π_F_ref[i]
            π_test = π_F_batch_log[i]
            
            # Avoid division by zero
            threshold = 1e-100
            non_zero_mask = (np.abs(π_ref) > threshold) | (np.abs(π_test) > threshold)
            if np.any(non_zero_mask):
                denom = np.maximum(np.abs(π_ref[non_zero_mask]), np.abs(π_test[non_zero_mask]))
                denom = np.maximum(denom, threshold)
                rel_errors = np.abs(π_ref[non_zero_mask] - π_test[non_zero_mask]) / denom
                max_rel_error = max(max_rel_error, float(np.max(rel_errors)))
        
        print(f"  F_batch_log (size={batch_size}) vs F: max rel error = {max_rel_error:.6e}")
        if max_rel_error < 1e-10:
            print(f"    ✓ Excellent agreement")
        elif max_rel_error < 1e-6:
            print(f"    ✓ Good agreement")
        elif max_rel_error < 1e-3:
            print(f"    ⚠ Acceptable agreement")
        else:
            print(f"    ✗ Significant disagreement")
    
    # Performance summary
    print(f"\n{'='*70}")
    print("Performance Summary:")
    print(f"{'Method':<30s} {'Time (s)':<15s} {'ms/call':<15s} {'Speedup':<15s}")
    print(f"{'-'*70}")
    baseline_time = results['F']['time']
    print(f"{'F (sequential)':<30s} {baseline_time:<15.4f} {results['F']['time_per_call']:<15.4f} {'1.00x':<15s}")
    
    for batch_size in batch_sizes:
        if batch_size > n_observations:
            continue
        key_batch = f'F_batch_{batch_size}'
        key_log = f'F_batch_log_{batch_size}'
        speedup_batch = baseline_time / results[key_batch]['time']
        speedup_log = baseline_time / results[key_log]['time']
        print(f"{f'F_batch (size={batch_size})':<30s} {results[key_batch]['time']:<15.4f} {results[key_batch]['time_per_call']:<15.4f} {f'{speedup_batch:.2f}x':<15s}")
        print(f"{f'F_batch_log (size={batch_size})':<30s} {results[key_log]['time']:<15.4f} {results[key_log]['time_per_call']:<15.4f} {f'{speedup_log:.2f}x':<15s}")
    
    print(f"{'='*70}\n")
    
    return results

# Run test
print("Testing with quantization_level=2 (2x2 maps, 16 total maps)")
results_F_comparison = test_F_batch_comparison(quantization_level=2, n_observations=1000, batch_sizes=[1, 10, 50, 100, 200, 500])


Testing with quantization_level=2 (2x2 maps, 16 total maps)
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_5c1c430d.npz

=== F, F_batch, and F_batch_log Comparison Test ===
Quantization level: 2
Map size: 2x2 = 4 cells
Total maps: 2^4 = 16
Number of observations: 1000
Batch sizes to test: [1, 10, 50, 100, 200, 500]

Generating test observations...

--- Testing Sequential F (one observation at a time) ---


  Time: 132.0827s (132.0827 ms/call)

--- Testing F_batch (batch_size=1) ---


  Time: 36.9666s (36.9666 ms/call)
  Speedup: 3.57x vs sequential
  Time per batch: 36.9666 ms

--- Testing F_batch (batch_size=10) ---


  Time: 3.8674s (3.8674 ms/call)
  Speedup: 34.15x vs sequential
  Time per batch: 38.6739 ms

--- Testing F_batch (batch_size=50) ---


  Time: 0.6493s (0.6493 ms/call)
  Speedup: 203.42x vs sequential
  Time per batch: 32.4653 ms

--- Testing F_batch (batch_size=100) ---


...

  Time: 3.5961s (3.5961 ms/call)
  Speedup: 36.73x vs sequential
  Time per batch: 35.9606 ms

--- Testing F_batch_log (batch_size=50) ---


  Time: 0.6565s (0.6565 ms/call)
  Speedup: 201.19x vs sequential
  Time per batch: 32.8248 ms

--- Testing F_batch_log (batch_size=100) ---


  Time: 0.3316s (0.3316 ms/call)
  Speedup: 398.35x vs sequential
  Time per batch: 33.1574 ms

--- Testing F_batch_log (batch_size=200) ---


  Time: 0.1715s (0.1715 ms/call)
  Speedup: 770.24x vs sequential
  Time per batch: 34.2966 ms

--- Testing F_batch_log (batch_size=500) ---


  Time: 0.0849s (0.0849 ms/call)
  Speedup: 1555.74x vs sequential
  Time per batch: 42.4502 ms

Numerical Agreement:
  F_batch (size=1) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch (size=10) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch (size=50) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch (size=100) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch (size=200) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch (size=500) vs F: max rel error = 7.337089e-14
    ✓ Excellent agreement
  F_batch_log (size=1) vs F: max rel error = 7.264779e-14
    ✓ Excellent agreement
  F_batch_log (size=10) vs F: max rel error = 7.264779e-14
    ✓ Excellent agreement
  F_batch_log (size=50) vs F: max rel error = 7.264779e-14
    ✓ Excellent agreement
  F_batch_log (size=100) vs F: max rel error = 7.264779e-14
    ✓ Excellent agreement
  F_batch_log (size=200) vs F: max rel error = 7.26477